In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, SeparableConv2D, BatchNormalization, Activation, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

In [2]:
# === CONFIGURATION ===
data_dir = 'D:/MSc. Asutosh/Final Year Project (Fetal Death Diagnosis Research)/Final Dataset/Dataset 1 - CTU CHB/CNN'
IMG_HEIGHT, IMG_WIDTH = 112, 336
BATCH_SIZE = 32
EPOCHS = 10

In [3]:
# === MesoNet Model ===
def build_meso_model(input_shape=(112, 336, 3), dropout=0.3, filters=16):
    input_layer = Input(shape=input_shape)

    x = SeparableConv2D(filters, (3, 3), padding='same')(input_layer)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2, 2), padding='same')(x)

    x = SeparableConv2D(filters, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2, 2), padding='same')(x)

    x = SeparableConv2D(32, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2, 2), padding='same')(x)

    x = SeparableConv2D(32, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(4, 4), padding='same')(x)

    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(dropout)(x)
    x = Dense(16, activation='relu')(x)
    x = Dropout(dropout)(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=input_layer, outputs=output)
    return model


In [4]:
# === IMAGE GENERATORS ===
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True,
    seed=42
)

val_gen = datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    seed=42
)


Found 4416 images belonging to 2 classes.
Found 1104 images belonging to 2 classes.


In [5]:
# === COMPUTE CLASS WEIGHTS ===
train_labels = train_gen.classes
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = dict(enumerate(weights))
print("Class Weights:", class_weights)

Class Weights: {0: 2.6285714285714286, 1: 0.6174496644295302}


In [6]:
# === BUILD & COMPILE MODEL ===
model = build_meso_model(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# === TRAIN ===
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    class_weight=class_weights
)


Epoch 1/10
138/138 [==============================] - 137s 979ms/step - loss: 0.7569 - accuracy: 0.5020 - val_loss: 0.6952 - val_accuracy: 0.1902
Epoch 2/10
138/138 [==============================] - 56s 402ms/step - loss: 0.6969 - accuracy: 0.6173 - val_loss: 0.6917 - val_accuracy: 0.8098
Epoch 3/10
138/138 [==============================] - 56s 402ms/step - loss: 0.6932 - accuracy: 0.6187 - val_loss: 0.6933 - val_accuracy: 0.1902
Epoch 4/10
138/138 [==============================] - 56s 401ms/step - loss: 0.6939 - accuracy: 0.6732 - val_loss: 0.6942 - val_accuracy: 0.1902
Epoch 5/10
138/138 [==============================] - 56s 402ms/step - loss: 0.6932 - accuracy: 0.6633 - val_loss: 0.6940 - val_accuracy: 0.1902
Epoch 6/10
138/138 [==============================] - 56s 402ms/step - loss: 0.6932 - accuracy: 0.7253 - val_loss: 0.6927 - val_accuracy: 0.8098
Epoch 7/10
138/138 [==============================] - 56s 403ms/step - loss: 0.6932 - accuracy: 0.3655 - val_loss: 0.6933 - val_a

In [ ]:
# === EVALUATION ===
val_gen.reset()
y_true = val_gen.classes
y_pred = model.predict(val_gen)
y_pred_labels = (y_pred > 0.5).astype(int).flatten()

print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred_labels, digits=4))

cm = confusion_matrix(y_true, y_pred_labels)
print("Confusion Matrix:\n", cm)

In [ ]:
# === PLOT ACCURACY/LOSS ===
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.show()
